# ДЗ 8. Мониторинг ML-систем

## 1. Ключевые метрики ML системы

Главная целевая метрика - выручка сервиса.

Бизнес метрики 1 уровня:
- Conversion Rate
- User Engagement
- Retention

ML метрики 2 уровня:
- Precision@K
- Recall@K
- Diversity
- Churn Rate

DevOps метрики 3 уровня:
- Latency p95
- Error Rate
- Availability 

Инфраструктурные метрики 4 уровня:
- RPS
- Inference Time
- Deploy Success Rate

## 2. Настройка мониторинга

Были настроены prometheus и graphana для мониторинга, а также написан простой ml сервис:

1. Поднимем всю необходимую инфраструктуру при помощи [docker-compose.yaml](docker-compose.yaml):

In [ ]:
!docker compose down
!docker compose up --build -d

2. На ml сервисе поддержана ручка metrics, для которой настроен мониторинг:

In [ ]:
!curl http://localhost:8000/metrics

3. График метрики в prometheus:

- http://localhost:9090/

<div style="text-align:center">
  <img src="assets/img_prometheus_graph.png" width="800">
</div>

4. Дешборд в graphana:

- http://localhost:3000/

<div style="text-align:center">
  <img src="assets/img_dashboard.png" width="800">
</div>

5. Настроенный алерт:

<div style="text-align:center">
  <img src="assets/img_alert_firing.png" width="800">
</div>

5. Нотификация об алерте в tg:

<div style="text-align:center">
  <img src="assets/img_alert_contact.png" width="800">
</div>

## 3. Обнаружение дрифта данных

Для отслеживания дрифта данных была поддержаны api/drift ручки.

1. Проверим статус дрифта сразу после включения сервиса:

In [ ]:
!curl http://localhost:8000/api/drift

2. Включаем дрифт через специальную ручку:

In [ ]:
!curl -X POST http://localhost:8000/api/drift/trigger

3. Визуализация детекции дрифта в graphana:

<div style="text-align:center">
  <img src="assets/img_data_drift.png" width="800">
</div>

## 4. Обеспечение качества данных

В задании предлагается настроить мониторинг качества данных при помощи dqops. Был добавлен соответствующий image в docker-compose, при запуске получаем следующую ошибку:

```txt
(.venv) mishasdk@m1-pro ~/m/d/mipt-ml-monitoring (main)> docker run -it -p 8080:8080 -v $(pwd)/dqops_workspace:/dqo/userhome dqops/dqo

  ___     ___     ___
 |   \   / _ \   / _ \   _ __   ___
 | |) | | (_) | | (_) | | '_ \ (_-<
 |___/   \__\_\  \___/  | .__/ /__/
                        |_|
 :: DQOps Data Quality Operations Center ::    (v1.13.1)

DQOps no longer supports a FREE edition. Please contact DQOps sales at https://dqops.com/contact-us/ to obtain a valid trial key.
```

Тут говорится что dqps больше нельзя использовать бесплатно, поэтому у нас не получится использовать его для решения задания.

Будем использовать альтернативу в виде Evidently, мы им уже пользовались в предыдущем задании для генерации отчетов по data drift. С помощью данного интрумента можно созадавать отчеты и смотреть результаты изменения распределений параметров и пропусков в данных, по мимо data drift. 

Открывается через http://localhost:8001/.

1. Сгенерированный отчет:

<div style="text-align:center">
  <img src="assets/img_eventialy_reports.png" width="800">
</div>

3. Открытый отчет, данные по data drift и распределению данных:

<div style="text-align:center">
  <img src="assets/img_eventialy_drift.png" width="800">
</div>

## 5. Cхема ML-системы для Virtual Product Placement

Схема построена в compose-diagram в [docker-compose-diagram.yml](docker-compose-diagram.yml):

In [ ]:
# Для отрисовки диаграммы нужно иметь установленный graphviz.
!compose-diagram --file docker-compose-diagram.yaml --direction=LR --nodesep=1.2

In [ ]:
import base64
from IPython.display import display, HTML

with open('docker-compose.png', 'rb') as f:
    data = base64.b64encode(f.read()).decode()
display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="800"></div>'))

Компоненты ML-системы:

- `ingest_gateway` — точка входа видеопотока, нарезка на кадры, публикация в `frames.raw`
- `compositor` — сборка обработанных кадров из `frames.final` в видеопоток, отдача клиенту
- `broker` — Kafka, соединяет все стадии пайплайна через стримы (publisher/subscriber)
- `person_detector` — дискриминативный ИИ: детекция людей и сегментация одежды на кадре
- `generative_service` — генеративный ИИ: inpainting логотипа на одежде
- `brand_catalog` — сервис каталога брендов и кампаний
- `brand_catalog_db` — хранилище логотипов и конфигов кампаний
- `mlinference` — единая точка инференса для всех ML-моделей
- `mlregistry` — версионирование и раздача моделей
- `mlregistry_db` — метаданные версий моделей
- `mltrain` — обучение и дообучение моделей
- `airflow` — оркестрация ML-пайплайнов
- `mltrain_db` — метаданные экспериментов
- `monitoring` — anti-deepfake guard, аудит согласий, технические метрики
- `monitoring_db` — журнал событий для аудита